# Kaggle runner — LLM reliability study
Full instructions: `docs/KAGGLE_STEPS.md` in the repo.

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**. Secrets (Add-ons → Secrets, attached to this notebook): `HF_TOKEN`, `GH_TOKEN`.

Every session: edit the **PLAN** cell, then **Save Version → Save & Run All**, and stop the interactive session. The run stops itself before the 12 h limit, and the zip in the last cell is the version's output. Next session: add that output as input (Add Input → Your Work → this notebook) so the restore cell carries finished generations forward.

In [ ]:
# 0. Secrets
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
GH = sec.get_secret('GH_TOKEN')

In [ ]:
# 1. PLAN — edit this cell each session
import time
T0 = time.time()
HOURS_BUDGET = 11.0                 # stop launching/kill runs after this; leaves time to score + zip
BACKEND = 'transformers'            # 'vllm' if cell 3 installed it and the timing check passed
RUN_M1 = True                       # timing + prefill pilot (first session only)
MMLU_MODELS = ['qwen2.5-3b', 'llama3.2-3b', 'phi3.5-mini', 'smollm3-3b', 'granite3.3-2b']
BFCL_N = 100                        # 100 = pilot, 200 = full
BFCL_MODELS = []                    # fill in once MMLU is done, e.g. the same five models

In [ ]:
# 2. Code (private repo) + packages
%cd /kaggle/working
!test -d llm-reliability || git clone -q https://{GH}@github.com/krisy1/llm-reliability.git
%cd /kaggle/working/llm-reliability
!git pull -q
!git log --oneline -1
!pip install -q -r requirements.txt
!pip freeze > /kaggle/working/pip_freeze.txt; mkdir -p results/logs; cp /kaggle/working/pip_freeze.txt results/logs/
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 3. Optional vLLM (only if BACKEND == 'vllm')
if BACKEND == 'vllm':
    !pip install -q vllm || echo 'vLLM failed -> set BACKEND = transformers'

In [ ]:
# 5. Items, reshuffles, manifests (deterministic: same files every session).
# Full manifests need the finalized wordings in data/reworded/; until then only the M1 pilot can run.
import pathlib
HAVE_MMLU_W = pathlib.Path('data/reworded/mmlu_pro_n300_wordings.jsonl').exists()
HAVE_BFCL_W = pathlib.Path(f'data/reworded/bfcl_multiple_n{BFCL_N}_wordings.jsonl').exists()
print('MMLU wordings:', HAVE_MMLU_W, '| BFCL wordings:', HAVE_BFCL_W)
!python -m src.sample_mmlu --n 300
!python -m src.make_variants orders --items data/sampled/mmlu_pro_n300.jsonl
if HAVE_MMLU_W:
    !python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --name mmlu300_full
!python -m src.sample_bfcl --n {BFCL_N}
!python -m src.make_variants orders --items data/sampled/bfcl_multiple_n{BFCL_N}.jsonl --config configs/experiment_bfcl.yaml
if HAVE_BFCL_W:
    !python -m src.build_manifest --config configs/experiment_bfcl.yaml --items data/sampled/bfcl_multiple_n{BFCL_N}.jsonl --name bfcl{BFCL_N}

In [ ]:
# 5. Items, reshuffles, manifests (deterministic: same files every session)
!python -m src.sample_mmlu --n 300
!python -m src.make_variants orders --items data/sampled/mmlu_pro_n300.jsonl
!python -m src.build_manifest --items data/sampled/mmlu_pro_n300.jsonl --seeds 0 1 2 3 --wordings w0 w1 w2 --name mmlu300_full
!python -m src.sample_bfcl --n {BFCL_N}
!python -m src.make_variants orders --items data/sampled/bfcl_multiple_n{BFCL_N}.jsonl --config configs/experiment_bfcl.yaml
!python -m src.build_manifest --config configs/experiment_bfcl.yaml --items data/sampled/bfcl_multiple_n{BFCL_N}.jsonl --name bfcl{BFCL_N}

In [ ]:
# 6. Runner: two lanes (one per T4), each runs its jobs in order; every job is killed at the deadline
import subprocess, threading, pathlib
pathlib.Path('results/logs').mkdir(parents=True, exist_ok=True)
def left(): return HOURS_BUDGET * 3600 - (time.time() - T0)

def job(model, manifest, config=None, extra=(), bs=32):
    cmd = ['python', '-m', 'src.run_generation', '--model', model, '--manifest', manifest,
           '--batch-size', str(bs), '--backend', BACKEND]
    if config: cmd += ['--config', config]
    return cmd + list(extra)

def lane(gpu, jobs):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    for cmd in jobs:
        if left() < 600: print(f'GPU{gpu}: no time left, skipping', cmd[4]); continue
        log = open(f'results/logs/gpu{gpu}.log', 'a'); print(f'GPU{gpu} start:', ' '.join(cmd[3:]), flush=True)
        p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
        try: p.wait(timeout=left())
        except subprocess.TimeoutExpired: p.kill(); p.wait(); print(f'GPU{gpu}: deadline, stopped (resumable)')
        print(f'GPU{gpu} done ({p.returncode}):', cmd[4], f'| {left()/3600:.1f} h left', flush=True)

def run_jobs(jobs):
    lanes = [jobs[0::2], jobs[1::2]]
    ts = [threading.Thread(target=lane, args=(g, js)) for g, js in enumerate(lanes)]
    [t.start() for t in ts]; [t.join() for t in ts]
    !tail -n 3 results/logs/gpu0.log results/logs/gpu1.log

In [ ]:
# 8. Full MMLU-Pro (9,000 runs per model)
if HAVE_MMLU_W and MMLU_MODELS:
    run_jobs([job(m, 'data/manifests/mmlu300_full.jsonl') for m in MMLU_MODELS])
else:
    print('skipped: no finalized MMLU wordings yet (or MMLU_MODELS empty)')

In [ ]:
# 9. BFCL multiple
if HAVE_BFCL_W and BFCL_MODELS:
    run_jobs([job(m, f'data/manifests/bfcl{BFCL_N}.jsonl', config='configs/experiment_bfcl.yaml', bs=16) for m in BFCL_MODELS])
else:
    print('skipped: no finalized BFCL wordings yet (or BFCL_MODELS empty)')

In [ ]:
# 9. BFCL multiple
if BFCL_MODELS:
    run_jobs([job(m, f'data/manifests/bfcl{BFCL_N}.jsonl', config='configs/experiment_bfcl.yaml', bs=16) for m in BFCL_MODELS])

In [ ]:
# 11. Score + analyse (works on whatever is finished; final numbers only when every model is COMPLETE)
if HAVE_MMLU_W:
    !python -m src.score_runs --manifest data/manifests/mmlu300_full.jsonl || true
    !python -m src.analyse --scored results/scored/mmlu300_full.csv --ranking-only || true
if HAVE_BFCL_W and BFCL_MODELS:
    !python -m src.score_runs --manifest data/manifests/bfcl{BFCL_N}.jsonl || true
    !python -m src.analyse --config configs/experiment_bfcl.yaml --scored results/scored/bfcl{BFCL_N}.csv --ranking-only || true

In [ ]:
# 11. Score + analyse (works on whatever is finished; final numbers only when every model is COMPLETE)
!python -m src.score_runs --manifest data/manifests/mmlu300_full.jsonl || true
!python -m src.analyse --scored results/scored/mmlu300_full.csv --ranking-only || true
if BFCL_MODELS:
    !python -m src.score_runs --manifest data/manifests/bfcl{BFCL_N}.jsonl || true
    !python -m src.analyse --config configs/experiment_bfcl.yaml --scored results/scored/bfcl{BFCL_N}.csv --ranking-only || true

In [ ]:
# 12. Save everything as this version's output
!rm -f /kaggle/working/results.zip
!zip -qr /kaggle/working/results.zip results data/manifests data/sampled
!ls -lh /kaggle/working/results.zip